In [1]:
# ================================================================
# 🚀 WIDS DATATHON - FINAL HIGH SCORE PIPELINE
# (Multi-seed + blending)
# ================================================================

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# ================================================================
# LOAD DATA
# ================================================================
BASE = "/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26"

train = pd.read_csv(BASE + "/train.csv")
test  = pd.read_csv(BASE + "/test.csv")

ID_COL = next((c for c in ["event_id","id","ID"] if c in test.columns), None)

TARGET = "time_to_hit_hours"
EVENT  = "event"

# ================================================================
# FEATURES
# ================================================================
features = [
    c for c in train.columns
    if c not in [ID_COL, TARGET, EVENT]
    and train[c].dtype != "object"
    and c in test.columns
]

def build_features(df):
    d = df[features].copy()

    dist = d["dist_min_ci_0_5h"].clip(lower=0)
    speed = d["closing_speed_m_per_h"].clip(lower=0)

    d["eta"] = np.where(speed > 1, dist / (speed + 1e-6), 200)
    d["risk"] = speed / (dist + 1)
    d["interaction"] = dist * speed

    d.fillna(0, inplace=True)
    return d

X = build_features(train).values
X_test = build_features(test).values

# ================================================================
# LABELS
# ================================================================
y_time = train[TARGET].values
y_event = train[EVENT].values

HORIZONS = [12, 24, 48, 72]
labels = {}

for h in HORIZONS:
    labels[h] = ((y_time <= h) & (y_event == 1)).astype(int)

# ================================================================
# TRAIN FUNCTION (STABLE ENSEMBLE)
# ================================================================
def train_model(X, y, X_test, seed):

    tscv = TimeSeriesSplit(n_splits=5)
    preds = np.zeros(len(X_test))

    for fold, (tr, va) in enumerate(tscv.split(X)):

        X_tr, X_val = X[tr], X[va]
        y_tr, y_val = y[tr], y[va]

        pos = sum(y_tr)
        neg = len(y_tr) - pos
        scale = neg / (pos + 1e-6)

        # MODELS
        xgb = XGBClassifier(
            n_estimators=850,
            learning_rate=0.03,
            max_depth=5,
            subsample=0.9,
            colsample_bytree=0.9,
            scale_pos_weight=scale,
            eval_metric="auc",
            random_state=seed
        )

        lgb = LGBMClassifier(
            n_estimators=850,
            learning_rate=0.03,
            num_leaves=64,
            class_weight="balanced",
            random_state=seed,
            verbosity=-1
        )

        cb = CatBoostClassifier(
            iterations=850,
            learning_rate=0.03,
            depth=6,
            verbose=0,
            random_state=seed
        )

        # TRAIN
        xgb.fit(X_tr, y_tr)
        lgb.fit(X_tr, y_tr)
        cb.fit(X_tr, y_tr)

        # PREDICT
        val_pred = (
            0.35*xgb.predict_proba(X_val)[:,1] +
            0.25*lgb.predict_proba(X_val)[:,1] +
            0.40*cb.predict_proba(X_val)[:,1]
        )

        test_pred = (
            0.35*xgb.predict_proba(X_test)[:,1] +
            0.25*lgb.predict_proba(X_test)[:,1] +
            0.40*cb.predict_proba(X_test)[:,1]
        )

        print(f"Seed {seed} Fold AUC:", roc_auc_score(y_val, val_pred))

        preds += test_pred / 5

    return np.clip(preds, 0.01, 0.99)

# ================================================================
# TRAIN MULTIPLE SEEDS
# ================================================================
seeds = [42, 99, 2024]
all_preds = {h: [] for h in HORIZONS}

for seed in seeds:
    print(f"\n🚀 TRAINING SEED {seed}")

    for h in HORIZONS:
        preds = train_model(X, labels[h], X_test, seed)
        all_preds[h].append(preds)

# ================================================================
# BLENDING
# ================================================================
final_preds = {}

for h in HORIZONS:
    p1, p2, p3 = all_preds[h]

    # weighted blend
    blended = 0.5*p1 + 0.3*p2 + 0.2*p3

    # small calibration
    blended = 0.99 * blended + 0.005

    final_preds[h] = np.clip(blended, 0.01, 0.99)

# ================================================================
# MONOTONIC FIX
# ================================================================
p = np.stack([final_preds[h] for h in HORIZONS], axis=1)
p = np.maximum.accumulate(p, axis=1)

for i, h in enumerate(HORIZONS):
    final_preds[h] = p[:, i]

# ================================================================
# SUBMISSION
# ================================================================
submission = pd.DataFrame({
    ID_COL: test[ID_COL],
    "prob_12h": final_preds[12],
    "prob_24h": final_preds[24],
    "prob_48h": final_preds[48],
    "prob_72h": final_preds[72]
})

submission.to_csv("final_submission.csv", index=False)

print("\n🚀 FINAL SUBMISSION READY (IMPROVED SCORE)")


🚀 TRAINING SEED 42
Seed 42 Fold AUC: 0.9852216748768473
Seed 42 Fold AUC: 0.9741935483870967
Seed 42 Fold AUC: 0.9464882943143813
Seed 42 Fold AUC: 0.9722222222222223
Seed 42 Fold AUC: 0.9508928571428572
Seed 42 Fold AUC: 1.0
Seed 42 Fold AUC: 0.9548387096774195
Seed 42 Fold AUC: 0.9675324675324676
Seed 42 Fold AUC: 1.0
Seed 42 Fold AUC: 1.0
Seed 42 Fold AUC: 1.0
Seed 42 Fold AUC: 1.0
Seed 42 Fold AUC: 1.0
Seed 42 Fold AUC: 1.0
Seed 42 Fold AUC: 1.0
Seed 42 Fold AUC: 1.0
Seed 42 Fold AUC: 1.0
Seed 42 Fold AUC: 1.0
Seed 42 Fold AUC: 1.0
Seed 42 Fold AUC: 1.0

🚀 TRAINING SEED 99
Seed 99 Fold AUC: 0.9753694581280787
Seed 99 Fold AUC: 0.9741935483870967
Seed 99 Fold AUC: 0.9464882943143813
Seed 99 Fold AUC: 0.9666666666666668
Seed 99 Fold AUC: 0.9464285714285715
Seed 99 Fold AUC: 1.0
Seed 99 Fold AUC: 0.9548387096774195
Seed 99 Fold AUC: 0.974025974025974
Seed 99 Fold AUC: 1.0
Seed 99 Fold AUC: 1.0
Seed 99 Fold AUC: 1.0
Seed 99 Fold AUC: 1.0
Seed 99 Fold AUC: 1.0
Seed 99 Fold AUC: 1.0
See